# 08 · MD quick checks: TM helicity, lid refit, zinc model

Standalone follow-up to notebook 07 (not a pipeline stage: the calculations live in this notebook, and it uses MDTraj).
It reads the same production trajectory and topology from Drive.

1. **TM helicity (residues 20–43)**: is the TM RMSD jump at the end of the run the helix pair moving as a unit, or unfolding?
2. **Lid vs 3BHX / 2XEG, refitted on 62–750**: does the closed-lid result change if the AlphaFold stalk residues 57–61 are left out of the fit?
3. **Zinc model type**: are the zincs *bonded* to their ligands in the topology (MCPB.py bonded model)?

Check the paths in the **CONFIG** cell, then *Runtime → Run all* (a CPU runtime is enough).

**Validation:** the loading cell must report the same frames as notebook 07, and the "57–750 (as before)" rows of
check 2 must reproduce notebook 07's lid RMSDs. Both held for the run recorded below.

In [ ]:
!pip -q install mdtraj
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
# ---------------- CONFIG ----------------
BASE = "/content/drive/MyDrive/psma_structural_modelling/outputs"
TOP = f"{BASE}/md_setup/system/system.prmtop"      # full system; the DCDs hold the solute, which comes first in it

TRAJ_DIR = f"{BASE}/md_run/traj"
TRAJ  = [f"{TRAJ_DIR}/prod_production_{i}.dcd" for i in range(1, 7)]        # 0-6 ns (prod_production_7 is an empty failed attempt)
TRAJ += [f"{TRAJ_DIR}/prod_t{t}.dcd" for t in range(6000, 13500, 250)]       # 6.0-13.5 ns; raise 13500 to include later segments
# keep this order: sort by the time in the name, not by file date

RMSD_CSV = f"{BASE}/md_analysis/rmsd_by_region.csv"   # from notebook 07, for the overlay plot (None to skip)

N_RES_PER_CHAIN = 750     # residues per chain in the model, UniProt 1-750, chain A first
SKIP_FRAMES = 0           # frames to drop at the start
FRAME_DT_NS = 0.01        # time between saved frames (ns)
FIRST_TIME_NS = 0.01      # time of the first production frame (ns)
print(len(TRAJ), "trajectory files")

In [ ]:
import numpy as np, pandas as pd, mdtraj as md, matplotlib.pyplot as plt

top_full = md.load_topology(TOP)
BB = {"N", "CA", "C", "O"}
# protein residues = residues with a full backbone (works with MCPB names such as AP1/HD1/GU1,
# and never picks up the calcium ion, whose atom is also called CA)
prot_res = [r for r in top_full.residues if BB <= {a.name for a in r.atoms}]
N = N_RES_PER_CHAIN
assert len(prot_res) == 2 * N, f"found {len(prot_res)} protein residues, expected {2 * N}: check N_RES_PER_CHAIN"

bb_idx = sorted(a.index for r in prot_res for a in r.atoms if a.name in BB)
traj = md.load(TRAJ, top=TOP, atom_indices=bb_idx)[SKIP_FRAMES:]    # backbone only: small in memory
t = FIRST_TIME_NS + FRAME_DT_NS * np.arange(traj.n_frames)
sub_res = list(traj.topology.residues)                                # same order as prot_res
assert len(sub_res) == 2 * N

def chain(i):   return "AB"[i // N]
def uniprot(i): return i % N + 1
def ca_of(i):   return next(a.index for a in sub_res[i].atoms if a.name == "CA")

print(f"{traj.n_frames} frames, {t[0]:.2f}-{t[-1]:.2f} ns", "" if traj.n_frames == 1350 else
      "  <- the earlier CSVs had 1350 frames: check SKIP_FRAMES / TRAJ")
print("Sanity check of the residue numbering (expect His/Asp/Glu variants and TRP at 541):")
for c, ch in enumerate("AB"):
    print(" chain", ch, {u: prot_res[c * N + u - 1].name for u in (377, 387, 425, 453, 541, 553)})

## Check 1 · TM helicity (residues 20–43)

DSSP per frame, per chain. `simplified=True` counts α-, 3₁₀- and π-helix together as helix (`H`).
Three residues of context on each side so the helix ends are assigned properly.

In [ ]:
helicity, per_res_before, per_res_end = {}, {}, {}
end = t >= 13.0                                   # the last 0.5 ns, where the TM RMSD jumped
for c, ch in enumerate("AB"):
    res_ids = [i for i in range(c * N, (c + 1) * N) if 17 <= uniprot(i) <= 46]
    atoms = [a.index for i in res_ids for a in sub_res[i].atoms]
    ss = md.compute_dssp(traj.atom_slice(atoms), simplified=True)        # (frames, residues)
    up = np.array([uniprot(i) for i in res_ids])
    core = (up >= 20) & (up <= 43)
    is_h = ss[:, core] == "H"
    helicity[ch] = is_h.mean(axis=1)
    per_res_before[ch] = pd.Series(is_h[~end].mean(axis=0), index=up[core])
    per_res_end[ch] = pd.Series(is_h[end].mean(axis=0), index=up[core])

blocks = [(0, 2.7), (2.7, 5.4), (5.4, 8.1), (8.1, 10.8), (10.8, 12), (12, 13), (13, 99)]
tab = pd.DataFrame({f"{a}-{min(b, t[-1]):.1f} ns": {f"TM helicity {ch}": helicity[ch][(t >= a) & (t < b)].mean()
                                                       for ch in "AB"} for a, b in blocks})
display(tab.round(2))
display(pd.DataFrame({f"{ch}, before 13 ns": per_res_before[ch] for ch in "AB"}
                     | {f"{ch}, last 0.5 ns": per_res_end[ch] for ch in "AB"}).round(2).T)

fig, ax = plt.subplots(figsize=(10, 3.5))
for ch, col in zip("AB", ["tab:blue", "tab:orange"]):
    ax.plot(t, pd.Series(helicity[ch]).rolling(10, center=True).mean(), color=col, label=f"helicity {ch} (0.1 ns mean)")
ax.set_ylim(0, 1.05); ax.set_xlabel("production time (ns)"); ax.set_ylabel("TM helicity (20–43)")
if RMSD_CSV:
    r = pd.read_csv(RMSD_CSV)
    ax2 = ax.twinx()
    for ch, col in zip("AB", ["tab:blue", "tab:orange"]):
        ax2.plot(r.iloc[:, 0], r[f"transmembrane helix (20–43), chain {ch}"], color=col, ls=":", lw=1, label=f"TM RMSD {ch}")
    ax2.set_ylabel("TM RMSD, ectodomain fit (Å)  [dotted]")
ax.legend(loc="lower left"); plt.tight_layout(); plt.show()

## Check 2 · Lid vs closed (3BHX) and open (2XEG), refitted

Each chain's Cα are superposed on the reference Cα (Kabsch fit, computed here), then the RMSD of the lid
(541–548) is measured *without* refitting. Done twice: with the fit range used before (57–750, 694 Cα)
and with the crystal ectodomain only (62–750).

In [ ]:
def ref_ca(pdb_id):
    ref = md.load_pdb(f"https://files.rcsb.org/download/{pdb_id}.pdb")
    ch0 = ref.topology.chain(0)
    return {a.residue.resSeq: (a.residue.name, ref.xyz[0, a.index] * 10) for a in ch0.atoms if a.name == "CA"}

def fit_rmsd(X, Y, fit, ev):
    # Kabsch superposition of every frame of X (F, n, 3) onto Y (n, 3) using atoms `fit`;
    # returns the RMSD (Å) of atoms `ev` after that superposition.
    P, Q = X[:, fit], Y[fit]
    pc, qc = P.mean(1, keepdims=True), Q.mean(0)
    H = np.einsum("fni,nj->fij", P - pc, Q - qc)
    U, S, Vt = np.linalg.svd(H)
    V, Ut = Vt.transpose(0, 2, 1), U.transpose(0, 2, 1)
    D = np.zeros((len(X), 3, 3)); D[:, 0, 0] = D[:, 1, 1] = 1; D[:, 2, 2] = np.sign(np.linalg.det(V @ Ut))
    R = V @ D @ Ut
    Xa = np.einsum("fij,fnj->fni", R, X[:, ev] - pc) + qc
    return np.sqrt(((Xa - Y[ev]) ** 2).sum(-1).mean(-1))

LID = list(range(541, 549))
FITS = {"57–750 (as before)": (57, 750), "62–750 (crystal only)": (62, 750)}
refs = {"closed": ("3BHX", ref_ca("3BHX")), "open": ("2XEG", ref_ca("2XEG"))}
rows, lid = [], {}
for state, (pdb_id, ref) in refs.items():
    for c, ch in enumerate("AB"):
        model = {uniprot(i): (sub_res[i].name, ca_of(i)) for i in range(c * N, (c + 1) * N)}
        if c == 0:   # numbering check: names should agree except His/MCPB-renamed residues
            bad = [(u, model[u][0], ref[u][0]) for u in sorted(set(model) & set(ref)) if model[u][0] != ref[u][0]]
            print(f"{pdb_id}: {len(bad)} residue-name mismatches with the model, e.g. {bad[:8]}")
        for fit_name, (lo, hi) in FITS.items():
            ids = [u for u in range(lo, hi + 1) if u in model and u in ref]
            X = traj.xyz[:, [model[u][1] for u in ids]] * 10
            Y = np.array([ref[u][1] for u in ids])
            fit = np.arange(len(ids)); ev = np.array([ids.index(u) for u in LID if u in ids])
            r = fit_rmsd(X, Y, fit, ev)
            lid[(ch, state, fit_name)] = r
            rows.append({"chain": ch, "reference": f"{pdb_id} ({state})", "fit": fit_name, "fit Cα": len(ids),
                         "fit RMSD (Å)": fit_rmsd(X, Y, fit, fit).mean(), "lid RMSD mean (Å)": r.mean(),
                         "lid RMSD last 50% (Å)": r[len(r) // 2:].mean()})
display(pd.DataFrame(rows).round(2))
print("Frames whose lid is nearer to the closed structure:")
for ch in "AB":
    for fit_name in FITS:
        frac = (lid[(ch, "closed", fit_name)] < lid[(ch, "open", fit_name)]).mean()
        print(f"  chain {ch}, fit {fit_name:<22} {100 * frac:5.1f} %")

## Check 3 · Zinc model: bonded or not?

In an MCPB.py *bonded* model the topology contains real bonds from each Zn to its ligand atoms.
In a non-bonded model (ions with only charge + Lennard-Jones) there are none.

In [ ]:
if top_full.n_bonds == 0:
    print("This topology file contains no bonds at all (e.g. PDB/GRO): use the .prmtop to check.")
zn = [a for a in top_full.atoms if (a.element is not None and a.element.symbol == "Zn") or a.name.upper().startswith("ZN")]
zn_idx = {a.index for a in zn}
partners = {a.index: [] for a in zn}
for a, b in top_full.bonds:
    for x, y in ((a, b), (b, a)):
        if x.index in zn_idx:
            partners[x.index].append(f"{y.residue.name}{y.residue.resSeq}:{y.name}")
for a in zn:
    p = partners[a.index]
    print(f"{a.residue.name}{a.residue.resSeq}:{a.name} (atom {a.index + 1})  {len(p)} bonds  {p}")
n = [len(v) for v in partners.values()]
print("\n=> BONDED zinc model" if n and min(n) >= 3 else "\n=> NON-BONDED zinc model (no Zn-ligand bonds)" if n and max(n) == 0
      else "\n=> mixed / unclear: look at the list above")

## Results of the 26 Nov 2026 run (1,350 frames, 0.01–13.50 ns)

**Check 1.** TM helicity 0.96–1.00 in both chains throughout, also in the last 0.5 ns (A 0.99, B 1.00). Only the ends
fray (chain B residue 20: 0.37 before 13 ns; chain A 41–43: 0.92–0.97). The late TM RMSD rise (to 11.8 / 13.1 Å) is a
**rigid-body displacement of the intact TM dimer** relative to the ectodomain, not unfolding.

**Check 2.** Lid RMSD to 3BHX (closed) / 2XEG (open): chain A 1.01 / 6.54 Å, chain B 1.29 / 6.46 Å, identical within
0.01 Å for the 57–750 (692 Cα) and 62–750 (687 Cα) fits. Nearest reference = closed in 100 % of frames, both chains.
Reproduces notebook 07 (1.02 / 6.59 and 1.30 / 6.51 Å).

**Check 3.** **Bonded** zinc model. Zn1: His377 NE2, Asp387 OD1, Asp453 OD2, OX1 (4 bonds). Zn2: Asp387 OD2,
Glu425 OE1 + OE2, His553 NE2, OX1 (5 bonds). The bonded distances are held by the force field (a setup check). The
**non-bonded** Zn1 contacts are the genuine observations: Asp453 OD1 (mean 2.26 Å) and Asp387 OD2 (2.33 Å) stay
coordinated; Glu425 OE2 drifts away (2.77 → 3.12 Å). MDTraj numbers residues from 0 in that list (HD1376 = His377).

**Still open:** the ectodomain tilt and the TM rigid-body motion are not converged at 13.5 ns. Rerun with the extended trajectory.

## How to read the results

**Check 1.** TM helicity stays high (≳ 0.8) in the last 0.5 ns → the TM RMSD jump is the **helix pair moving as a unit**
(hinging at the stalk). Helicity drops at the same time as the RMSD rises → part of the helix **unfolds**; the per-residue
table shows where (often at the membrane–water ends, 20–23 or 40–43).

**Check 2.** The two fits give lid RMSDs within ~0.1–0.2 Å and the lid stays nearer to 3BHX in ~100 % of frames →
the closed-lid conclusion does not depend on the fit; report the 62–750 fit in the methods. A larger difference means the
stalk residues were pulling the superposition, and the 62–750 numbers are the ones to use.
If the name-mismatch line shows almost every residue mismatched, the reference numbering is offset: tell me.

**Check 3.** Bonded → the stable Zn–ligand distances are largely built into the force field; present them as a check that
the setup is sound, not as a result. Non-bonded → the intact site is a genuine finding (and a good one for 13.5 ns).